In [ ]:
#To obtain the RoBERTa model quickly, switch to GPU

#Steps:
#1. Go to Runtime
#2. Change runtime type → T4 GPU
#3. Will disconnect your runtime if u are already connected -> Save

#With GPU, RoBERTa runs ~20x faster, bringing it down to ~15-20 minutes - else the model could take up to 5 hours

In [ ]:
#Mount Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
#Set Base Path
BASE = '/content/drive/MyDrive/Text Miners/Actual Work'
print('Connected:', BASE)

Connected: /content/drive/MyDrive/Text Miners/Actual Work


In [ ]:
#Confirm connection
import os
print(os.listdir(BASE))

['data', 'pre-processing', 'features', 'task1-sentiments', 'task2-topics', 'task3-combination', 'dashboard', 'BeforeYouStart.docx']


In [ ]:
#Load Data
import pandas as pd
import pickle

# Load labelled reviews
df = pd.read_csv(f'{BASE}/data/labelled_reviews.csv')
print("Shape:", df.shape)
print("Label distribution:\n", df['label'].value_counts())
print(df.head(3))

Shape: (380505, 4)
Label distribution:
 label
positive    359559
neutral      14784
negative      6162
Name: count, dtype: int64
   listing_id                                           comments  \
0       27934  We stayed in the apartment for a week and we e...   
1       27934  My girlfriend and I recently stayed in Nuttee'...   
2       27934  I stayed for one month at the condo and was re...   

   compound_score     label  
0          0.9729  positive  
1          0.9875  positive  
2          0.9913  positive  


In [ ]:
#Load TF-IDF Matrix
with open(f'{BASE}/features/tfidf_matrix.pkl', 'rb') as f:
    tfidf_matrix = pickle.load(f)

print("TF-IDF shape:", tfidf_matrix.shape)
print("Number of labelled reviews:", len(df))

TF-IDF shape: (380505, 10000)
Number of labelled reviews: 380505


In [ ]:
#Split train and test data
from sklearn.model_selection import train_test_split

X = tfidf_matrix
y = df['label']  #change 'label' if the column is named differently

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y  #preserves class balance
)

print("Train size:", X_train.shape[0])
print("Test size:", X_test.shape[0])
print("Test label distribution:\n", y_test.value_counts())

Train size: 304404
Test size: 76101
Test label distribution:
 label
positive    71912
neutral      2957
negative     1232
Name: count, dtype: int64


In [ ]:
#Train Logistics Regression + Generate classification report
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

lr_model = LogisticRegression(C=1.0, max_iter=1000)
lr_model.fit(X_train, y_train)

y_pred_lr = lr_model.predict(X_test)

print("=== Logistic Regression Classification Report ===")
print(classification_report(y_test, y_pred_lr,
      target_names=['negative', 'neutral', 'positive']))

=== Logistic Regression Classification Report ===
              precision    recall  f1-score   support

    negative       0.67      0.50      0.57      1232
     neutral       0.73      0.46      0.56      2957
    positive       0.98      0.99      0.98     71912

    accuracy                           0.97     76101
   macro avg       0.79      0.65      0.71     76101
weighted avg       0.96      0.97      0.96     76101



In [ ]:
#Run RoBERTa on same test set
from transformers import pipeline

# Load pretrained RoBERTa sentiment model
roberta = pipeline(
    "text-classification",
    model="cardiffnlp/twitter-roberta-base-sentiment",
    truncation=True,
    max_length=512
)

# Get the actual review texts for the test set
test_texts = df.iloc[y_test.index]['comments'].tolist()  # change 'comments' if needed

print("Running RoBERTa on", len(test_texts), "reviews... (this will take a while)")

roberta_results = roberta(test_texts, batch_size=32)
print("Done!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/747 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

Running RoBERTa on 76101 reviews... (this will take a while)
Done!


In [ ]:
#Generate classification report for RoBERTa
#Map RoBERTa labels to match the labelling scheme
label_map = {
    'LABEL_0': 'negative',
    'LABEL_1': 'neutral',
    'LABEL_2': 'positive'
}

y_pred_roberta = [label_map[r['label']] for r in roberta_results]

print("=== RoBERTa Classification Report ===")
print(classification_report(y_test, y_pred_roberta,
      target_names=['negative', 'neutral', 'positive']))

=== RoBERTa Classification Report ===
              precision    recall  f1-score   support

    negative       0.48      0.78      0.59      1232
     neutral       0.46      0.28      0.34      2957
    positive       0.98      0.98      0.98     71912

    accuracy                           0.95     76101
   macro avg       0.64      0.68      0.64     76101
weighted avg       0.95      0.95      0.95     76101



In [ ]:
#Side by side comparision of both models
from sklearn.metrics import f1_score

f1_lr = f1_score(y_test, y_pred_lr, average='weighted')
f1_roberta = f1_score(y_test, y_pred_roberta, average='weighted')

print("=== F1 Score Comparison ===")
print(f"Logistic Regression (TF-IDF): {f1_lr:.4f}")
print(f"RoBERTa (pretrained):         {f1_roberta:.4f}")

=== F1 Score Comparison ===
Logistic Regression (TF-IDF): 0.9618
RoBERTa (pretrained):         0.9474


In [ ]:
# TASK 1.1 RESULTS ANALYSIS

# DATASET CONTEXT
# - Heavy class imbalance: num. of positive (94.5%), neutral (3.9%), negative (1.6%)
# - This inflates overall accuracy and weighted F1 for both models
# - Macro avg F1 is the more honest metric here (treats all classes equally)

# MODEL COMPARISON SUMMARY
# ┌──────────┬──────────┬─────────┬──────────┬──────────────┐
# │  Class   │  LR F1   │ RoBERTa │  Winner  │    Notes     │
# ├──────────┼──────────┼─────────┼──────────┼──────────────┤
# │ negative │  0.57    │  0.59   │ RoBERTa  │ Small gap    │
# │ neutral  │  0.56    │  0.34   │ LR       │ Large gap    │
# │ positive │  0.98    │  0.98   │ Tie      │              │
# │ macro    │  0.71    │  0.64   │ LR       │              │
# │ weighted │  0.96    │  0.95   │ LR       │              │
# └──────────┴──────────┴─────────┴──────────┴──────────────┘

# KEY FINDINGS
# 1. LR outperforms RoBERTa overall despite being a simpler model.
#    Weighted F1: LR 0.9618 vs RoBERTa 0.9474

# 2. RoBERTa is marginally better at detecting negatives (0.59 vs 0.57).
#    Likely because negative reviews use strong distinctive language that transformers trained on social media recognise well.

# 3. RoBERTa performs poorly on neutral class (F1 = 0.34).
#    Likely due to domain mismatch — cardiffnlp/twitter-roberta-base-sentiment was trained on tweets, which rarely contain neutral/mild language.
#    Airbnb neutral reviews ("it was okay, nothing special") don't match the tweet tone the model learned from.

# 4. Both models are genuinely weak on minority classes (negative, neutral).
#    This is a class imbalance problem, not a model quality problem.

# CONCLUSION
# LR on TF-IDF is the stronger model for this dataset.
# However, a model specifically fine-tuned on hotel/accommodation reviews would likely outperform both, especially on neutral and negative classes.